# Module 2 — The Division of Labour and the Pin Factory
## Specialisation, Productivity, and the Gains from Trade

**From Smith to Simulation: Computing the Ideas that Built Economics**  
The University of Edinburgh · School of Economics

---

## Part I — The History

### The Most Famous Factory in Economics

*The Wealth of Nations* opens not with grand theory but with a visit to a **pin factory**. Smith describes watching workers make pins — the thin metal fasteners used in clothing and textiles — and marvels at how dividing the work into separate tasks transforms productivity:

> *"One man draws out the wire, another straights it, a third cuts it, a fourth points it, a fifth grinds it at the top for receiving the head; to make the head requires two or three distinct operations..."*  
> — Adam Smith, *The Wealth of Nations*, Book I, Chapter 1

Smith counted **eighteen distinct operations** in making a pin. Ten workers, each specialising in a few tasks, could produce about **48,000 pins per day** — roughly 4,800 per worker. But if each worker tried to do everything alone, Smith estimated they could make at most **20 pins per day**, perhaps not even one.

That is a **240-fold increase** in productivity from specialisation alone.

### Why Does Specialisation Work?

Smith identified three reasons:

1. **Skill improvement.** A worker who performs one task all day becomes extraordinarily good at it — faster, more precise, more consistent.

2. **Time saving.** Workers don't waste time switching between tasks, moving between workstations, or picking up different tools.

3. **Innovation.** Workers focused on a single task are more likely to invent machines or methods that improve it. Smith noted that many early industrial machines were invented by workers themselves, not by scientists.

### From Pin Factories to Nations: Comparative Advantage

Smith extended the logic of specialisation from factories to countries. Just as workers should specialise in what they do best, nations should trade with each other rather than trying to produce everything domestically.

This idea was refined forty years later by **David Ricardo** (1772–1823), who proved something remarkable: even if one country is better at producing *everything*, both countries still gain from trade. What matters is not **absolute advantage** (who's better overall) but **comparative advantage** (who gives up less to produce each good).

> *"Under a system of perfectly free commerce, each country naturally devotes its capital and labour to such employments as are most beneficial to each."*  
> — David Ricardo, *Principles of Political Economy and Taxation* (1817), Chapter 7

This is one of the most counterintuitive and important results in all of economics, and we'll compute it in this module.

### Edinburgh Connection

The pin factory Smith described was likely based on factories he observed in or near Edinburgh and Kirkcaldy. In the 1760s, Scotland's manufacturing was booming — linen, iron, and metalwork — and Edinburgh's merchants were deeply embedded in international trade networks. The Old Town's closes and wynds were full of workshops where the division of labour was already visible in miniature. Smith's genius was to see in these ordinary workshops a universal principle.

---

## Part II — The Computation

### Setting Up

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Simulating the Pin Factory

Let's start by simulating Smith's pin factory. We'll compare two modes of production:

1. **No specialisation:** Each worker performs all 18 steps to make a complete pin.
2. **Division of labour:** Workers specialise in a subset of steps.

We'll model this simply: each step takes some amount of time. Without specialisation, a worker moves through all steps slowly (switching costs, less skill). With specialisation, each step takes less time (practice, no switching).

In [ ]:
# Pin factory simulation
n_steps = 18  # number of distinct operations

# Time per step (minutes) WITHOUT specialisation
# A generalist is slow at everything
time_per_step_generalist = 3.0  # 3 minutes per step
switching_cost = 1.0            # 1 minute lost switching between tasks

# Time per step WITH specialisation (practice makes perfect)
time_per_step_specialist = 0.5  # 30 seconds per step — 6x faster through practice
switching_cost_specialist = 0.0 # no switching — one task all day

# Working day: 10 hours = 600 minutes
working_day = 600

# GENERALIST: time to make one pin = all steps + switching
time_per_pin_generalist = n_steps * time_per_step_generalist + (n_steps - 1) * switching_cost
pins_per_day_generalist = working_day / time_per_pin_generalist

print("=== No Specialisation (Generalist) ===")
print(f"Time per pin: {time_per_pin_generalist:.1f} minutes")
print(f"Pins per worker per day: {pins_per_day_generalist:.1f}")

# SPECIALIST: 10 workers, each handles ~2 steps
n_workers = 10
steps_per_worker = n_steps / n_workers

# The bottleneck is the slowest worker (worker with the most steps)
# With 18 steps and 10 workers, some handle 2, some handle 1
# But in a pipeline, throughput = 1 pin per bottleneck_time
bottleneck_time = np.ceil(steps_per_worker) * time_per_step_specialist
pins_per_day_total = working_day / bottleneck_time
pins_per_day_specialist = pins_per_day_total  # total output of the team
pins_per_worker_specialist = pins_per_day_total / n_workers

print(f"\n=== Division of Labour ({n_workers} Specialist Workers) ===")
print(f"Bottleneck time per pin: {bottleneck_time:.1f} minutes")
print(f"Total pins per day: {pins_per_day_total:.0f}")
print(f"Pins per worker per day: {pins_per_worker_specialist:.0f}")

print(f"\n=== Productivity Gain ===")
print(f"Specialisation multiplier: {pins_per_worker_specialist / pins_per_day_generalist:.0f}x")
print(f"Smith's estimate: 240x")

### How Productivity Grows with the Number of Specialists

Smith noted that the division of labour is "limited by the extent of the market." A tiny village can't support 10 specialist pin-makers — there aren't enough customers. Specialisation only works at scale. Let's see how output per worker changes as we add more workers to the factory:

In [ ]:
def pins_per_worker(n_workers, n_steps=18, t_specialist=0.5, t_generalist=3.0,
                    switch_cost=1.0, day=600):
    """
    Compute output per worker as the workforce grows.
    With 1 worker: generalist.
    With n workers: divide steps among them (pipeline).
    """
    if n_workers == 1:
        time_per_pin = n_steps * t_generalist + (n_steps - 1) * switch_cost
        return day / time_per_pin
    
    # Each worker handles ceil(n_steps / n_workers) steps
    # But no worker handles more than n_steps
    steps_each = max(1, n_steps / min(n_workers, n_steps))
    
    # Specialist speed improves with focus (fewer steps = more practice)
    # Model: time per step = t_specialist * (steps_each / 1)^0.3 (learning curve)
    effective_time = t_specialist * steps_each**0.3
    
    bottleneck = np.ceil(steps_each) * effective_time
    total_pins = day / bottleneck
    return total_pins / n_workers

workers_range = np.arange(1, 51)
output = [pins_per_worker(n) for n in workers_range]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(workers_range, output, 'o-', color='steelblue', markersize=4, linewidth=1.5)
ax.axvline(18, color='red', linestyle='--', alpha=0.5, label='18 workers (one per step)')
ax.set_xlabel('Number of workers', fontsize=12)
ax.set_ylabel('Pins per worker per day', fontsize=12)
ax.set_title("Smith's Insight: Productivity Rises with Specialisation", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Output per worker rises steeply as workers specialise in fewer steps.")
print("Beyond 18 workers (one per step), gains plateau — the division of labour")
print("is limited by the number of distinct tasks.")

### Comparative Advantage: Ricardo's Great Insight

Now let's move from factories to nations. Ricardo's theory of **comparative advantage** shows that trade benefits both countries even when one is better at producing everything.

Consider two countries — **Scotland** and **Portugal** — producing two goods: **cloth** and **wine**.

| | Hours to produce 1 unit of cloth | Hours to produce 1 unit of wine |
|---|---|---|
| Scotland | 100 | 120 |
| Portugal | 90 | 80 |

Portugal is better at producing *both* goods (absolute advantage in everything). A naive analysis might suggest Portugal has nothing to gain from trading with Scotland. Ricardo proved this wrong.

In [ ]:
# Labour costs (hours per unit)
scotland_cloth = 100
scotland_wine = 120
portugal_cloth = 90
portugal_wine = 80

# Opportunity cost: to make 1 unit of cloth, how much wine do you give up?
opp_cost_cloth_scotland = scotland_cloth / scotland_wine  # cloth in terms of wine
opp_cost_cloth_portugal = portugal_cloth / portugal_wine

opp_cost_wine_scotland = scotland_wine / scotland_cloth  # wine in terms of cloth
opp_cost_wine_portugal = portugal_wine / portugal_cloth

print("Opportunity costs:")
print(f"\n  Scotland: 1 cloth costs {opp_cost_cloth_scotland:.3f} wine")
print(f"  Portugal: 1 cloth costs {opp_cost_cloth_portugal:.3f} wine")
print(f"\n  Scotland: 1 wine costs {opp_cost_wine_scotland:.3f} cloth")
print(f"  Portugal: 1 wine costs {opp_cost_wine_portugal:.3f} cloth")

print(f"\nScotland has comparative advantage in: ", end='')
if opp_cost_cloth_scotland < opp_cost_cloth_portugal:
    print("CLOTH (lower opportunity cost)")
else:
    print("WINE (lower opportunity cost)")

print(f"Portugal has comparative advantage in: ", end='')
if opp_cost_wine_portugal < opp_cost_wine_scotland:
    print("WINE (lower opportunity cost)")
else:
    print("CLOTH (lower opportunity cost)")

Scotland gives up less wine to produce cloth (0.833 vs 1.125 units), so Scotland has a comparative advantage in cloth. Portugal gives up less cloth to produce wine, so Portugal has a comparative advantage in wine.

### The Production Possibility Frontier

Each country has a fixed amount of labour. The **Production Possibility Frontier (PPF)** shows all combinations of cloth and wine a country can produce. Let's say each country has 1,000 hours of labour available.

In [ ]:
total_hours = 1000  # each country has 1000 labour hours

# PPF: if you produce Q_cloth units of cloth, you can produce at most:
# Q_wine = (total_hours - cost_cloth * Q_cloth) / cost_wine

def ppf(Q_cloth, cost_cloth, cost_wine, hours=total_hours):
    """Maximum wine given cloth production."""
    hours_left = hours - cost_cloth * Q_cloth
    return np.maximum(hours_left / cost_wine, 0)

# Maximum production of each good
scot_max_cloth = total_hours / scotland_cloth
scot_max_wine = total_hours / scotland_wine
port_max_cloth = total_hours / portugal_cloth
port_max_wine = total_hours / portugal_wine

cloth_range_s = np.linspace(0, scot_max_cloth, 100)
cloth_range_p = np.linspace(0, port_max_cloth, 100)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(cloth_range_s, ppf(cloth_range_s, scotland_cloth, scotland_wine),
             color='steelblue', linewidth=2.5)
axes[0].fill_between(cloth_range_s, ppf(cloth_range_s, scotland_cloth, scotland_wine),
                     alpha=0.1, color='steelblue')
axes[0].set_xlabel('Cloth', fontsize=12)
axes[0].set_ylabel('Wine', fontsize=12)
axes[0].set_title('Scotland\'s PPF', fontsize=13)
axes[0].set_xlim(0, 14)
axes[0].set_ylim(0, 14)
axes[0].grid(True, alpha=0.3)

axes[1].plot(cloth_range_p, ppf(cloth_range_p, portugal_cloth, portugal_wine),
             color='coral', linewidth=2.5)
axes[1].fill_between(cloth_range_p, ppf(cloth_range_p, portugal_cloth, portugal_wine),
                     alpha=0.1, color='coral')
axes[1].set_xlabel('Cloth', fontsize=12)
axes[1].set_ylabel('Wine', fontsize=12)
axes[1].set_title('Portugal\'s PPF', fontsize=13)
axes[1].set_xlim(0, 14)
axes[1].set_ylim(0, 14)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Production Possibility Frontiers', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Scotland can produce at most {scot_max_cloth:.1f} cloth or {scot_max_wine:.1f} wine.")
print(f"Portugal can produce at most {port_max_cloth:.1f} cloth or {port_max_wine:.1f} wine.")

### Computing the Gains from Trade

Without trade (autarky), suppose each country splits its labour equally between the two goods. With trade, each specialises in its comparative advantage and they exchange.

In [ ]:
# AUTARKY: each country splits labour 50/50
half_hours = total_hours / 2

scot_cloth_autarky = half_hours / scotland_cloth
scot_wine_autarky = half_hours / scotland_wine
port_cloth_autarky = half_hours / portugal_cloth
port_wine_autarky = half_hours / portugal_wine

world_cloth_autarky = scot_cloth_autarky + port_cloth_autarky
world_wine_autarky = scot_wine_autarky + port_wine_autarky

print("=== AUTARKY (no trade, 50/50 split) ===")
print(f"Scotland: {scot_cloth_autarky:.2f} cloth, {scot_wine_autarky:.2f} wine")
print(f"Portugal: {port_cloth_autarky:.2f} cloth, {port_wine_autarky:.2f} wine")
print(f"World:    {world_cloth_autarky:.2f} cloth, {world_wine_autarky:.2f} wine")

# SPECIALISATION: Scotland produces only cloth, Portugal produces only wine
scot_cloth_trade = total_hours / scotland_cloth  # all labour on cloth
scot_wine_trade = 0
port_cloth_trade = 0
port_wine_trade = total_hours / portugal_wine  # all labour on wine

world_cloth_trade = scot_cloth_trade + port_cloth_trade
world_wine_trade = scot_wine_trade + port_wine_trade

print(f"\n=== SPECIALISATION + TRADE ===")
print(f"Scotland produces: {scot_cloth_trade:.2f} cloth, {scot_wine_trade:.2f} wine")
print(f"Portugal produces: {port_cloth_trade:.2f} cloth, {port_wine_trade:.2f} wine")
print(f"World total:       {world_cloth_trade:.2f} cloth, {world_wine_trade:.2f} wine")

print(f"\n=== GAINS FROM TRADE ===")
print(f"Extra cloth: {world_cloth_trade - world_cloth_autarky:.2f} units")
print(f"Extra wine:  {world_wine_trade - world_wine_autarky:.2f} units")
print(f"\nSpecialisation creates MORE of BOTH goods — the world is richer.")

In [ ]:
# Visualise: combined PPF vs autarky
fig, ax = plt.subplots(figsize=(8, 6))

# Individual PPFs in autarky
ax.plot(cloth_range_s, ppf(cloth_range_s, scotland_cloth, scotland_wine),
        color='steelblue', linewidth=1.5, linestyle='--', alpha=0.6, label='Scotland PPF')
ax.plot(cloth_range_p, ppf(cloth_range_p, portugal_cloth, portugal_wine),
        color='coral', linewidth=1.5, linestyle='--', alpha=0.6, label='Portugal PPF')

# World PPF with trade (kinked line)
# First: Portugal makes all wine (up to 12.5), Scotland transitions from wine to cloth
# Then: Scotland makes all cloth
world_cloth = np.array([0, scot_max_cloth, scot_max_cloth + port_max_cloth])
world_wine = np.array([scot_max_wine + port_max_wine, port_max_wine, 0])
ax.plot(world_cloth, world_wine, 'k-', linewidth=2.5, label='World PPF (with trade)')

# Mark autarky and trade points
ax.plot(world_cloth_autarky, world_wine_autarky, 's', color='red', markersize=12,
        zorder=5, label='World output (autarky)')
ax.plot(world_cloth_trade, world_wine_trade, '*', color='gold', markersize=18,
        markeredgecolor='black', zorder=5, label='World output (trade)')

ax.set_xlabel('Total cloth', fontsize=12)
ax.set_ylabel('Total wine', fontsize=12)
ax.set_title('Gains from Trade: The World Produces More of Both Goods', fontsize=13)
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 25)
ax.set_ylim(0, 25)
plt.tight_layout()
plt.show()

### Terms of Trade: Splitting the Gains

Specialisation creates a surplus. But who gets it? That depends on the **terms of trade** — the price at which cloth is exchanged for wine. Both countries benefit as long as the exchange rate lies between their opportunity costs.

In [ ]:
print("For trade to benefit BOTH countries, the price of cloth (in wine) must lie between:")
print(f"  Scotland's opportunity cost: {opp_cost_cloth_scotland:.3f} wine per cloth")
print(f"  Portugal's opportunity cost: {opp_cost_cloth_portugal:.3f} wine per cloth")
print(f"\nAny price between {opp_cost_cloth_scotland:.3f} and {opp_cost_cloth_portugal:.3f} works.")

# Example: trade at 1 cloth = 1 wine
terms_of_trade = 1.0  # 1 cloth exchanges for 1 wine

# Scotland exports cloth, imports wine
# Scotland produces 10 cloth, keeps 5, trades 5 for 5 wine
scot_keeps_cloth = 5
scot_exports_cloth = scot_cloth_trade - scot_keeps_cloth
scot_gets_wine = scot_exports_cloth * terms_of_trade

print(f"\n=== Trade at {terms_of_trade} wine per cloth ===")
print(f"Scotland: has {scot_keeps_cloth:.1f} cloth + {scot_gets_wine:.1f} wine")
print(f"  (vs autarky: {scot_cloth_autarky:.2f} cloth + {scot_wine_autarky:.2f} wine)")
print(f"  Gain: +{scot_keeps_cloth - scot_cloth_autarky:.2f} cloth, +{scot_gets_wine - scot_wine_autarky:.2f} wine")

port_gets_cloth = scot_exports_cloth
port_keeps_wine = port_wine_trade - scot_gets_wine

print(f"\nPortugal: has {port_gets_cloth:.1f} cloth + {port_keeps_wine:.1f} wine")
print(f"  (vs autarky: {port_cloth_autarky:.2f} cloth + {port_wine_autarky:.2f} wine)")
print(f"  Gain: +{port_gets_cloth - port_cloth_autarky:.2f} cloth, +{port_keeps_wine - port_wine_autarky:.2f} wine")

print(f"\nBoth countries are better off — Ricardo's comparative advantage in action.")

---

## Part III — Exercises

### Exercise 1 — The Extent of the Market

Smith wrote that "the division of labour is limited by the extent of the market." A village of 100 people can't sustain a specialist pin-maker.

**(a)** Suppose a town consumes $D = 0.5 \times \text{population}$ pins per day. A generalist worker makes 10 pins/day; a specialist in a team of 10 makes 480 pins/day per team (4,800 total). If the minimum team for specialisation is 10 workers, what is the minimum population that justifies a specialist pin factory?

**(b)** Plot the cost per pin under each mode (generalist cost = wage / 10; specialist cost = 10 × wage / 4800) as a function of daily demand. At what demand level does specialisation become cheaper? Assume wage = 1.

**(c)** Smith observed that specialisation is most extreme in large cities and near seaports. Explain why in 2–3 sentences using the model.

In [ ]:
# Your answer here


### Exercise 2 — Three-Country Comparative Advantage

Extend Ricardo's model to three countries and two goods:

| | Hours per cloth | Hours per wine |
|---|---|---|
| Scotland | 100 | 120 |
| Portugal | 90 | 80 |
| France | 110 | 90 |

Each country has 1,000 hours of labour.

**(a)** Compute the opportunity cost of cloth (in terms of wine) for all three countries. Which has the strongest comparative advantage in cloth? In wine?

**(b)** Under autarky (50/50 split), what is world output of each good?

**(c)** Under full specialisation (each country produces only its comparative-advantage good — but with three countries and two goods, one country must split), find the allocation that maximises world output. (Hint: rank countries by opportunity cost of cloth.)

**(d)** Compute the gains from trade.

In [ ]:
# Your answer here


### Exercise 3 — Learning by Doing

Smith's first reason for the benefits of specialisation is that practice improves skill. Model this with a **learning curve**: the time to perform a task falls with repetition.

$$t(n) = t_0 \cdot n^{-\beta}$$

where $t(n)$ is the time for the $n$-th repetition, $t_0$ is the time for the first attempt, and $\beta > 0$ is the learning rate.

**(a)** A generalist performs each of 18 tasks 30 times per day. A specialist performs 1 task 540 times per day. Using $t_0 = 5$ minutes and $\beta = 0.3$, compute the average time per task for each worker.

**(b)** Plot $t(n)$ for $n = 1$ to $1000$ repetitions. When does the learning curve flatten out?

**(c)** Compute total pins produced per day for a 10-worker team with learning effects vs a single generalist with learning effects. How does the learning curve amplify the gains from specialisation?

In [ ]:
# Your answer here


---

## Part IV — Quiz

### Conceptual Questions

**Q1.** Smith identified three reasons why the division of labour increases productivity. Which of these is NOT one of them?

(a) Workers become more skilled through repetition  
(b) Workers save time by not switching between tasks  
(c) Workers are paid lower wages when they specialise  
(d) Specialised workers are more likely to invent improvements

**Q2.** Smith's pin factory showed that 10 workers with division of labour could produce roughly:

(a) 10 times more than 10 generalists  
(b) 50 times more than 10 generalists  
(c) 240 times more per worker than a generalist  
(d) The same amount, but at lower cost

**Q3.** "The division of labour is limited by the extent of the market" means:

(a) Markets should be regulated to prevent over-specialisation  
(b) Specialisation only pays off when there are enough customers to buy the output  
(c) Large markets lead to monopolies  
(d) Division of labour only works in manufacturing, not agriculture

**Q4.** Comparative advantage means:

(a) A country can produce a good more cheaply in absolute terms  
(b) A country gives up less of another good to produce one unit of this good  
(c) A country has more natural resources  
(d) A country has lower wages

**Q5.** Portugal has absolute advantage in both cloth and wine, but Scotland has comparative advantage in cloth. According to Ricardo, what should happen?

(a) Portugal should produce both goods and Scotland should import everything  
(b) Scotland should specialise in cloth and Portugal in wine, and they should trade  
(c) Neither country should trade since Portugal is better at both  
(d) Scotland should adopt Portuguese technology

### Computational Questions

**Q6.** If Scotland takes 100 hours to produce 1 cloth and 120 hours to produce 1 wine, the opportunity cost of 1 cloth is:

(a) 100/120 = 0.833 wine  
(b) 120/100 = 1.2 wine  
(c) 100 wine  
(d) 20 wine

**Q7.** A country with 1,000 labour hours, needing 50 hours per cloth and 100 hours per wine, can produce at most:

(a) 20 cloth or 10 wine  
(b) 10 cloth or 20 wine  
(c) 50 cloth or 100 wine  
(d) 20 cloth or 20 wine

**Q8.** The PPF (Production Possibility Frontier) is a straight line when:

(a) There are increasing returns to scale  
(b) The opportunity cost is constant (no diminishing returns)  
(c) Both goods use the same inputs  
(d) The country doesn't trade

**Q9.** For trade to benefit both countries, the terms of trade must lie:

(a) Above both countries' opportunity costs  
(b) Below both countries' opportunity costs  
(c) Between the two countries' opportunity costs  
(d) Equal to the average of their opportunity costs

**Q10.** Full specialisation with trade increases world output because:

(a) Each country produces what it's relatively best at, minimising total resource use  
(b) Trade increases the total number of workers  
(c) Specialisation eliminates diminishing returns  
(d) Both countries adopt the same technology

### Quiz Answers

<details>
<summary>Click to reveal answers</summary>

**Q1.** (c) Lower wages is not one of Smith's reasons. His three reasons are: skill improvement through repetition, time saved by not switching tasks, and innovation by focused workers.

**Q2.** (c) Smith estimated a generalist could make about 20 pins/day, while a specialist in a 10-person team made about 4,800/day — a 240-fold increase per worker.

**Q3.** (b) You need a large enough market (enough customers) to justify the fixed costs of setting up specialised production. A village can't support a pin factory.

**Q4.** (b) Comparative advantage is about opportunity cost — what you give up. A country has comparative advantage in the good it can produce at the lowest opportunity cost.

**Q5.** (b) Ricardo showed that even when one country has absolute advantage in everything, both gain from trade if each specialises in its comparative advantage.

**Q6.** (a) To produce 1 cloth (100 hours), Scotland forgoes 100/120 = 0.833 units of wine that could have been made with those hours.

**Q7.** (a) 1000/50 = 20 cloth, or 1000/100 = 10 wine.

**Q8.** (b) A straight-line PPF means the opportunity cost doesn't change as you shift production — each unit of cloth always costs the same amount of wine. This happens with constant returns and a single input (labour).

**Q9.** (c) If the price is between their opportunity costs, each country gets a better deal than producing the good domestically.

**Q10.** (a) Specialisation allocates resources to their most efficient use globally, so the same total labour produces more of both goods.

</details>

---

## Further Reading

- Smith, A. *The Wealth of Nations* (1776), Book I, Chapters 1–3.
- Ricardo, D. *Principles of Political Economy and Taxation* (1817), Chapter 7.
- Heilbroner, R. *The Worldly Philosophers*, Chapter 3.
- Krugman, P. "Ricardo's Difficult Idea" (1996) — a short essay on why people find comparative advantage so hard to accept.

---

*Next week: Malthus, Ricardo, and the Limits to Growth — population dynamics, diminishing returns, and the Malthusian trap.*